In [8]:
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [7]:
pip install --upgrade jupyter ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 2.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd

df_prsa = pd.read_csv("/Users/xinsun/Desktop/econometrics 25 fall/term paper/PRSA_data_2010.1.1-2014.12.31.csv",encoding = 'latin1')  # 根据你的真实文件名改一下
df_new = pd.read_csv("/Users/xinsun/Desktop/econometrics 25 fall/term paper/new.csv",encoding = 'latin1')


/var/folders/8y/qmrfc1p10m107hhmw47kl5780000gn/T/ipykernel_22708/3041851789.py:4: DtypeWarning: Columns (1,11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_new = pd.read_csv("/Users/xinsun/Desktop/econometrics 25 fall/term paper/new.csv",encoding = 'latin1')


In [7]:
df_new.head()


,url,id,Lng,Lat,Cid,tradeTime,DOM,followers,totalPrice,price,...,buildingType,constructionTime,renovationCondition,buildingStructure,ladderRatio,elevator,fiveYearsProperty,subway,district,communityAverage
0,https://bj.lianjia.com/chengjiao/101084782030....,101084782030,116.475489,40.019520,1111027376244,2016-08-09,1464.0,106,415.0,31680,...,1.0,2005,3,6,0.217,1.0,0.0,1.0,7,56021.0
1,https://bj.lianjia.com/chengjiao/101086012217....,101086012217,116.453917,39.881534,1111027381879,2016-07-28,903.0,126,575.0,43436,...,1.0,2004,4,6,0.667,1.0,1.0,0.0,7,71539.0
2,https://bj.lianjia.com/chengjiao/101086041636....,101086041636,116.561978,39.877145,1111040862969,2016-12-11,1271.0,48,1030.0,52021,...,4.0,2005,3,6,0.500,1.0,0.0,0.0,7,48160.0
3,https://bj.lianjia.com/chengjiao/101086406841....,101086406841,116.438010,40.076114,1111043185817,2016-09-30,965.0,138,297.5,22202,...,1.0,2008,1,6,0.273,1.0,0.0,0.0,6,51238.0
4,https://bj.lianjia.com/chengjiao/101086920653....,101086920653,116.428392,39.886229,1111027381174,2016-08-28,927.0,286,392.0,48396,...,4.0,1960,2,2,0.333,0.0,1.0,1.0,1,62588.0


In [8]:
df_prsa.head()

,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
0,1,2010,1,1,0,NaN,-21,-11.0,1021.0,NW,1.79,0,0
1,2,2010,1,1,1,NaN,-21,-12.0,1020.0,NW,4.92,0,0
2,3,2010,1,1,2,NaN,-21,-11.0,1019.0,NW,6.71,0,0
3,4,2010,1,1,3,NaN,-21,-14.0,1019.0,NW,9.84,0,0
4,5,2010,1,1,4,NaN,-20,-12.0,1018.0,NW,12.97,0,0


In [12]:
daily_pm25=(
    df_prsa.groupby(['year','month','day'])['pm2.5']
    .mean()
    .reset_index()
)
daily_pm25.head()

,year,month,day,pm2.5
0,2010,1,1,NaN
1,2010,1,2,145.958333
2,2010,1,3,78.833333
3,2010,1,4,31.333333
4,2010,1,5,42.458333


In [13]:
df_new['tradeTime']=pd.to_datetime(df_new['tradeTime'])
df_new['tradeTime'].min(), df_new['tradeTime'].max()

(Timestamp('2002-06-01 00:00:00'), Timestamp('2018-01-28 00:00:00'))

In [15]:
start=pd.to_datetime('2010-01-01')
end=pd.to_datetime('2014-12-31')
mask=(df_new['tradeTime']>=start) & (df_new['tradeTime']<=end)

df_2010_2014=df_new.loc[mask].copy()
df_2010_2014.head()

,url,id,Lng,Lat,Cid,tradeTime,DOM,followers,totalPrice,price,...,buildingType,constructionTime,renovationCondition,buildingStructure,ladderRatio,elevator,fiveYearsProperty,subway,district,communityAverage
92216,https://bj.lianjia.com/chengjiao/BJ0000614981....,BJ0000614981,116.122491,39.939735,1111027377936,2010-01-01,1.0,0,165.0,13699,...,3.0,2008,4,6,0.333,1.0,0.0,0.0,12,39994.0
92217,https://bj.lianjia.com/chengjiao/BJ0000614985....,BJ0000614985,116.122150,39.932268,1111027378707,2010-01-05,1.0,0,72.5,10546,...,4.0,1998,3,2,0.500,0.0,0.0,0.0,12,35181.0
92218,https://bj.lianjia.com/chengjiao/BJ0000614991....,BJ0000614991,116.111318,39.949921,1111027377794,2010-01-15,1.0,0,114.0,12676,...,4.0,2003,3,2,0.500,0.0,0.0,0.0,12,36923.0
92219,https://bj.lianjia.com/chengjiao/BJ0000614992....,BJ0000614992,116.119651,39.934504,1111027382024,2010-01-16,1.0,0,84.0,11711,...,4.0,2005,3,6,0.333,1.0,0.0,0.0,12,39294.0
92220,https://bj.lianjia.com/chengjiao/BJ0000614999....,BJ0000614999,116.121964,39.939762,1111027375862,2010-01-18,1.0,0,80.0,12190,...,4.0,2003,3,2,0.500,0.0,0.0,0.0,12,37588.0


In [18]:
print('house price: ',df_2010_2014.shape)
print('pm2.5: ',daily_pm25.shape)

house price:  (114773, 26)
pm2.5:  (1826, 4)


In [19]:
df_2010_2014['tradeTime'].duplicated().any()

np.True_

In [20]:
df_2010_2014['tradeTime'].duplicated().sum()

np.int64(113309)

In [21]:
counts = df_2010_2014["tradeTime"].value_counts()
dup_dates = counts[counts > 1]   # 只保留出现次数>1 的日期

dup_dates.head()

tradeTime
2013-03-10    483
2013-03-03    447
2014-10-12    421
2013-03-09    383
2014-10-07    365
Name: count, dtype: int64